## Notebook43

### Setup

Run all of the following before starting the notebook.

In [ ]:
! wget -q -nc https://raw.githubusercontent.com/taylor-arnold/fds2/refs/heads/main/funs.py

In [ ]:
! pip install umap-learn --quiet
! python -m spacy download en_core_web_sm
! python -m spacy download fr_core_news_sm

In [ ]:
import polars as pl
from plotnine import ggplot, aes
from polars import col as c
import numpy as np

import funs
from funs import DSText, umap_text

import spacy

ub = "https://raw.githubusercontent.com/taylor-arnold/fds2/refs/heads/main/"

In [ ]:
meta = pl.read_csv(ub + "data/wiki_uk_meta.csv.gz")
text_en = pl.read_csv(ub + "data/wiki_uk_authors_text.csv")
text_fr = pl.read_csv(ub + "data/wiki_uk_authors_text_fr.csv")

In [ ]:
nlp_en = spacy.load("en_core_web_sm")
nlp_fr = spacy.load("fr_core_news_sm")

### Questions

`meta` holds one row per author, with a birth and death year, a literary
`era`, a `gender`, and the `link`/`short` names used elsewhere in the corpus.
`text_en` has one row per author with the full text of their English
Wikipedia page; `text_fr` is the same for the French edition. This is the
75-author corpus the chapter uses for its cross-language section, because it
is the only corpus in the catalogue with French text paired to it. The first
question deals with a narrowing this notebook has to make before anything
else: `text_fr` does not have 75 rows. Print results unless a question says
otherwise.

1. How many of the 75 authors in `text_en` also have a row in `text_fr`?
Anti-join to find the author(s) missing a French page, and join the result
to `meta` for their era and gender.

Now rank all 75 authors by the character length of their English page, and
report where the two missing authors fall in that ranking.

Only 73 of the 75 authors have a French Wikipedia page. Both authors
missing one, Rex Warner and Edward Upward, are Twentieth Century writers,
but era alone does not explain the gap, since plenty of other Twentieth
Century authors in this corpus do have a French page. Page length fits
better. Rex Warner has the shortest English page in the 75-author corpus, and
Edward Upward has the third shortest, so the two authors with no French page
are also two of the three least-covered on the English side. With only two
cases this is suggestive, not proof, but missing French coverage here
appears to track obscurity, not time period.

2. Restrict `text_en` and `text_fr` to the 73 shared authors and sort both
by `doc_id`. Then run each language's text through its own spaCy pipeline
with `DSText.process`, and print the resulting shapes.

3. For each language, compute the average number of alphabetic tokens per
document and the average number of distinct lemmas per document. In the
first block, do this for English (`anno_en`); in the second, for French
(`anno_fr`).

An English page runs about 5,184 alphabetic tokens on average, drawn from
about 1,446 distinct lemmas; a French page for the same author runs about
2,579 tokens from about 822 distinct lemmas. The token count halves, matching
the character-length gap between the two text columns (about 31,400
characters against 16,200). The lemma count falls less, to about 57% of the
English figure, because a shorter text repeats fewer of its words. French
Wikipedia covers these authors, but consistently more briefly than English
Wikipedia.

4. Compute the TF-IDF table for each language with
`DSText.compute_tfidf(..., min_df=0.01, max_df=0.5)`, as the chapter does,
and save them as `tfidf_en` and `tfidf_fr`. In the first block, print the
top 10 English terms for `"Jane Austen"`, `"A. A. Milne"`, and `"T. S.
Eliot"`, joined into a single string per author the same way the chapter
does. In the second block, do the same for French. Do the lists pick out
what each author is known for? Does anything look out of place?

Mostly yes. The English lists are clean: Austen gets her own name, her
sister Cassandra, and her novels; Milne gets Pooh, Winnie, and Christopher
Robin; Eliot gets The Waste Land, Prufrock, and the Quartets. But Milne's and
Eliot's English lists both contain `parser` (and Eliot's also has `output`),
words with no connection to a children's author or a modernist poet. The
French lists pick out many of the same titles (Mansfield Park, Northanger
Abbey, The Waste Land, Prufrock), but they are noisier, with stray words such
as `with`, `je`, and `quel` surviving the cutoff, and Milne's French list
keeps `Winnie` but loses Pooh and Christopher Robin. None of the French lists
contains `parser`. The next question tracks that down.

5. In the first block, count how many times the substring
`"mw-parser-output"` appears in each English page with
`.str.count_matches()`, and list the pages with any matches, sorted by
count. In the second block, run the same count on `text_fr` and print its
total. In the third block, filter `anno_en` to the token `"parser"` and
check its `upos` tag and `is_alpha` flag.

`"mw-parser-output"` is leaked CSS from a Wikipedia coordinate template,
scraped into the article body along with the rest of the page text. It
occurs 33 times across four English pages, mostly Eliot's and Milne's, and
never in the French text. The source is a run of repeated
`.mw-parser-output .geo-...` class names from a hidden geo-coordinate box.
spaCy tags the token `parser` as an ordinary alphabetic `NOUN`, just as it
would tag a real word, so `is_alpha` does not catch it the way it caught
notebook41's backslash artifact. It passes through `compute_tfidf` and puts a
scraping artifact into two authors' top-term lists.

6. Reproduce the chapter's part-of-speech comparison (stack `anno_en` and
`anno_fr` with a `lang` column, group by `lang` and `upos`, and compute the
proportion of each part of speech within each language), but now on the
matched 73-author subset both tables share. Report the determiner (`DET`)
proportion for each language.

8.9% of English tokens are determiners against 12.2% of French ones, the
same gap the chapter describes. This time the comparison is not confounded
by a population mismatch. The chapter stacks all 75 English pages against
only 73 French ones, while `anno_en` and `anno_fr` here cover the same 73
authors on both sides.

7. Compute the pairwise angle distance for each language separately with
`DSText.compute_distances(..., min_df=0.01, max_df=0.5)`. For each language,
stack the result with a copy of itself with the ID columns swapped, and find
every author's single nearest neighbor.

8. What proportion of authors have the exact same nearest neighbor under
both languages? Compare this to the chance rate of matching by accident
among 72 other authors.

Almost 48% of authors (35 of 73) have the same nearest neighbor whether the
distance is computed from their English page or their French one, over
thirty times the roughly 1.4% expected from two independent random guesses
among 72 other authors. The two languages are far from producing identical
geometry, but the structure that TF-IDF and the angle distance recover is
not arbitrary either: it survives translation for close to half the
corpus.

9. Build a 2D UMAP projection for each language with `umap_text(...,
min_df=0.01, max_df=0.5, random_state=42)`, join each author's short name
and `era` from `meta`, and plot both, colored by `era`. In the first block,
do this for English (`anno_en`); in the second, for French (`anno_fr`). Do
authors from the same era group together in either language?

Not in any clear way. In both plots the colors are scattered across the
whole space, with Twentieth Century names next to Restoration and Early ones
and no region belonging to a single era. Question 8 showed that each
language's geometry has real neighborhoods, since the nearest neighbor
survives translation far more often than chance, but era is not what
organizes them. Authors end up near each other because their pages share
specific names, places, and titles, not because they wrote in the same
century.

10. Looking back across this notebook: what carried over unchanged between
English and French, and what genuinely differed?

The machinery carried over unchanged: the same `DSText.process`,
`compute_tfidf`, `compute_distances`, and `umap_text` calls ran on French
tokens as they did on English ones, because none of that code has
language-specific logic. What differed was the input each pipeline received
and what came out. Two authors have no French page, and the gap appears to
track obscurity, not era. French pages run about half the length of their
English counterparts, with a little over half the vocabulary. French uses
determiners more often, a grammatical fact unrelated to editorial choice. The
English pages also carried a scraping artifact the French ones did not. And
the two languages' document geometries agree on an author's closest match
less than half the time, though far above chance. These are the kinds of
cross-lingual differences that, as the chapter's closing paragraph says, this
machinery lets you study systematically instead of asserting from a
hunch.